## 8. Comparaison des méthodes avec plus d'employés

On compare ici les deux méthodes développées pour un nombre d'employés plus important et un plus grand nombre de semaines (mais toujours 21 shifts par semaine). On appelle modele_v4.jl car le S_max est par semaine et dans modele_v3 il est global (donc problème insoluble). Ici les données générées sont conformes au droit pour les restrictions de suivi de shift en tiennent compte ainsi que les temps de travail sur la semaine. 

In [1]:
import Pkg
Pkg.activate("..")

using JuMP
using HiGHS
using Random
using Statistics
include("../src/modele_v4.jl")
include("../src/donnees.jl")
include("../src/generer_donnees.jl")
include("../src/recuit.jl")

data = generer_donnees(100; graine = 1, conforme = true);

# On vérifie si le planning caché vérifie les nouvelles règles
controleContraintes(data.planning_cache, data)

true

In [2]:
function resoudre(data; p_min = 1, p_semaine = 1)
    model, x, e, viol_type, viol_jour, viol_semaine =
        construire_modele(data; p_min, p_semaine)
    set_silent(model)
    optimize!(model)
    return (; model, x, e, viol_type, viol_jour, viol_semaine)
end

resoudre (generic function with 1 method)

In [3]:
model, x, e, viol_type, viol_jour, viol_semaine = resoudre(data; p_min = 1, p_semaine = 1)

println("Statut : ", termination_status(model))
println("Coût : ", objective_value(model))
println("Temps HiGHS : ", round(solve_time(model), digits = 2), " s")

Statut : OPTIMAL
Coût : 385.0
Temps HiGHS : 1.73 s


In [4]:
# On vérifie le planning HiGHS est bien valide
controleContraintes(x, data)

true

In [5]:
Random.seed!(1337)
x_depart, nb_essais = construire_planning(data)

if x_depart === nothing
    println("Aucun planning valide trouvé")
else
    println("Essais nécessaires : ", nb_essais)
    println("Coût de départ : ", calcul_cout(x_depart, data)[1])
    println("Valide : ", controleContraintes(x_depart, data))
end

recuit_simule2(x_depart, data, 10.0, 0.95, 1, 1)  # compilation, résultat ignoré

t = @elapsed recuit_simule2(x_depart, data, 10.0, 0.95, 600, 1)
println(round(t / 600 * 1000, digits = 2), " ms par itération")

Essais nécessaires : 7019
Coût de départ : 1934
Valide : true
34.9 ms par itération


L'algorithme glouton a nécessité 7019 essais contre 2 dans le cas de règles non conformes au droit. Les règles plus strictes rendent la construction aléatoire plus compliquée. 

Si le recuit arrive à trouver une solution optimale, ça sera en mettant nettement plus de temps qu'avec HiGHS. Ca n'est pas un outil envisageable pour nous. Si on reprend les paramètres trouvés dans le notebook précédent, 600 itérations et 90 paliers soit 54000 itérations de 35ms chacune. Cela ferait plus de 30 min (par extrapolation) quand HiGHS ne met que 1,72 secondes. 

Sur ce modèle, HiGHS est la bonne méthode de résolution.